# Extração de dados — Pesquisadores da UNIFEI (todos os institutos, separados)

Coleta, via API do OpenAlex, todos os pesquisadores de **todos os institutos da UNIFEI**,
classificando cada um no seu instituto (com base no texto de afiliação de cada trabalho),
mantendo o mesmo intervalo de tempo do script original: **2020-01-01 a 2026-08-19**.

Institutos considerados (Campus Itajubá + Campus Itabira):
- IEM — Instituto de Engenharia Mecânica
- IEPG — Instituto de Engenharia de Produção e Gestão
- IESTI — Instituto de Engenharia de Sistemas e Tecnologia da Informação
- IFQ — Instituto de Física e Química
- IMC — Instituto de Matemática e Computação
- IRN — Instituto de Recursos Naturais
- ISEE — Instituto de Sistemas Elétricos e Energia
- IEI — Instituto de Engenharias Integradas (Campus Itabira)

> Um mesmo pesquisador pode aparecer com mais de um instituto ao longo dos anos
> (mudou de instituto, atua em dois, etc.) — nesse caso todos aparecem na coluna `instituto`,
> separados por `;`. Quem não bater com nenhuma palavra-chave fica como `Não identificado`
> (você pode revisar a coluna `afiliacoes` desses casos e ajustar as palavras-chave abaixo).


In [ ]:
import requests
import pandas as pd
import time

# ============================================================
# CONFIGURAÇÃO
# ============================================================

BASE_URL = "https://api.openalex.org"

INSTITUTION_ID = "I8165072"  # UNIFEI

# Mesmo intervalo de tempo do script original
DATA_INICIO = "2020-01-01"
DATA_FIM = "2026-08-19"

# ============================================================
# PALAVRAS-CHAVE DE CADA INSTITUTO DA UNIFEI
# ============================================================
# Ajuste/complete aqui se algum instituto não estiver batendo direito.

INSTITUTOS_UNIFEI = {
    "IEM": [
        "Instituto de Engenharia Mecânica",
        "Institute of Mechanical Engineering",
    ],
    "IEPG": [
        "Instituto de Engenharia de Produção e Gestão",
        "Instituto de Produção e Gestão",
        "Institute of Production Engineering and Management",
        "Institute of Industrial Engineering and Management",
    ],
    "IESTI": [
        "Instituto de Engenharia de Sistemas e Tecnologia da Informação",
        "Institute of Systems Engineering and Information Technology",
    ],
    "IFQ": [
        "Instituto de Física e Química",
        "Institute of Physics and Chemistry",
    ],
    "IMC": [
        "Instituto de Matemática e Computação",
        "Institute of Mathematics and Computing",
    ],
    "IRN": [
        "Instituto de Recursos Naturais",
        "Institute of Natural Resources",
    ],
    "ISEE": [
        "Instituto de Sistemas Elétricos e Energia",
        "Institute of Electrical Systems and Energy",
        "Institute of Electrical Systems",
    ],
    "IEI": [
        "Instituto de Engenharias Integradas",
        "Institute of Integrated Engineering",
    ],
}


# ============================================================
# FUNÇÃO PARA BUSCAR TODOS OS TRABALHOS DA UNIFEI NO PERÍODO
# ============================================================

def buscar_trabalhos_unifei(institution_id, data_inicio, data_fim):

    url = f"{BASE_URL}/works"

    params = {
        "filter": (
            f"institutions.id:{institution_id},"
            f"from_publication_date:{data_inicio},"
            f"to_publication_date:{data_fim}"
        ),
        "per-page": 200,
        "cursor": "*"
    }

    todos = []

    while params["cursor"]:

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print("Erro:", response.status_code)
            print(response.text)
            break

        data = response.json()

        resultados = data.get("results", [])

        todos.extend(resultados)

        print(f"Trabalhos coletados: {len(todos)}")

        params["cursor"] = (
            data.get("meta", {})
                .get("next_cursor")
        )

        if not resultados:
            break

        time.sleep(0.1)

    return todos


print(f"Buscando trabalhos da UNIFEI entre {DATA_INICIO} e {DATA_FIM}...")

trabalhos = buscar_trabalhos_unifei(
    INSTITUTION_ID,
    DATA_INICIO,
    DATA_FIM
)

print("\nTotal de trabalhos:", len(trabalhos))


Buscando trabalhos da UNIFEI entre 2020-01-01 e 2026-08-19...
Trabalhos coletados: 200
Trabalhos coletados: 400
Trabalhos coletados: 600
Trabalhos coletados: 800
Trabalhos coletados: 1000
Trabalhos coletados: 1200
Trabalhos coletados: 1400
Trabalhos coletados: 1600
Trabalhos coletados: 1800
Trabalhos coletados: 2000
Trabalhos coletados: 2200
Trabalhos coletados: 2400
Trabalhos coletados: 2600
Trabalhos coletados: 2800
Trabalhos coletados: 3000
Trabalhos coletados: 3200
Trabalhos coletados: 3400
Trabalhos coletados: 3600
Trabalhos coletados: 3800
Trabalhos coletados: 4000
Trabalhos coletados: 4200
Trabalhos coletados: 4400
Trabalhos coletados: 4600
Trabalhos coletados: 4800
Trabalhos coletados: 5000
Trabalhos coletados: 5200
Trabalhos coletados: 5360

Total de trabalhos: 5360


In [ ]:
def identificar_institutos(texto_afiliacao):
    """Retorna a lista de siglas de institutos cujo nome aparece no texto."""

    texto = texto_afiliacao.lower()

    encontrados = []

    for sigla, palavras in INSTITUTOS_UNIFEI.items():
        if any(palavra.lower() in texto for palavra in palavras):
            encontrados.append(sigla)

    return encontrados


# ============================================================
# IDENTIFICAR TODOS OS PESQUISADORES E CLASSIFICAR POR INSTITUTO
# ============================================================

pesquisadores_unifei = {}

for work in trabalhos:

    ano = work.get("publication_year")

    for authorship in work.get("authorships", []):

        author = authorship.get("author")

        if not author:
            continue

        author_id = author.get("id")

        if not author_id:
            continue

        author_id = author_id.split("/")[-1]

        author_name = author.get("display_name")

        # Só mantemos o autor se a UNIFEI estiver entre as
        # instituições listadas nesse authorship
        institution_ids = [
            inst.get("id", "").split("/")[-1]
            for inst in authorship.get("institutions", [])
        ]

        if INSTITUTION_ID not in institution_ids:
            continue

        afiliacoes = authorship.get("raw_affiliation_strings", [])
        texto_afiliacao = " ".join(afiliacoes)

        institutos_encontrados = identificar_institutos(texto_afiliacao)

        if author_id not in pesquisadores_unifei:
            pesquisadores_unifei[author_id] = {
                "author_id": author_id,
                "name": author_name,
                "anos": set(),
                "afiliacoes": set(),
                "institutos": set()
            }

        pesquisadores_unifei[author_id]["anos"].add(ano)

        for afiliacao in afiliacoes:
            pesquisadores_unifei[author_id]["afiliacoes"].add(afiliacao)

        for sigla in institutos_encontrados:
            pesquisadores_unifei[author_id]["institutos"].add(sigla)


# ============================================================
# TRANSFORMAR EM DATAFRAME
# ============================================================

dados = []

for autor in pesquisadores_unifei.values():

    institutos = sorted(autor["institutos"])

    dados.append({
        "author_id": autor["author_id"],
        "name": autor["name"],
        "instituto": "; ".join(institutos) if institutos else "Não identificado",
        "anos": ", ".join(map(str, sorted(a for a in autor["anos"] if a))),
        "afiliacoes": " | ".join(sorted(autor["afiliacoes"]))
    })


unifei_df = pd.DataFrame(dados)

unifei_df = (
    unifei_df
    .sort_values(["instituto", "name"])
    .reset_index(drop=True)
)

print(
    f"Pesquisadores da UNIFEI (todos os institutos) entre {DATA_INICIO} e {DATA_FIM}:",
    len(unifei_df)
)

print("\nPesquisadores por instituto:")
print(unifei_df["instituto"].value_counts())

display(unifei_df)


Pesquisadores da UNIFEI (todos os institutos) entre 2020-01-01 e 2026-08-19: 3938

Pesquisadores por instituto:
instituto
Não identificado     2780
IRN                   234
IFQ                   221
IEM                   182
ISEE                  127
IEPG                  124
IESTI                  87
IMC                    78
IEI                    63
IEM; IRN                5
IEM; IMC                5
IFQ; IRN                5
IESTI; IMC              4
IEI; IEPG               4
IESTI; ISEE             3
IEM; IEPG               3
IEM; IFQ                2
IRN; ISEE               2
IEPG; IESTI; ISEE       2
IFQ; ISEE               2
IEI; IFQ                1
IEM; IFQ; IRN           1
IEPG; ISEE              1
IEM; IRN; ISEE          1
IEPG; IFQ               1
Name: count, dtype: int64


,author_id,name,instituto,anos,afiliacoes
0,A5000276221,ADRIELLE CRISTINA SASSA,IEI,"2022, 2023","Institute of Integrated Engineering, Federal U..."
1,A5070765202,Alisson Clementino Da Silva,IEI,2021,Laboratório Interdisciplinar de Materiais Comp...
2,A5045443140,Ana Carolina Moreira Silva,IEI,"2020, 2021",Instituto de Engenharias Integradas (IEI) da U...
3,A5039088323,Ana Carolina Oliveira Santos,IEI,"2020, 2021, 2022, 2024","Federal University of Itajuba, BPS Avenue, 130..."
4,A5033167027,Ana Cristina Tolentino Cabral,IEI,"2020, 2021, 2023",Interdisciplinary Laboratory of Advanced Mater...
...,...,...,...,...,...
3933,A5013407442,Ícaro Meidem Silva,Não identificado,2023,Unifei | Universidade Federal de Itajubá (UNIF...
3934,A5086891205,Ícaro Souza Ribeiro,Não identificado,2020,Universidade Federal de Itajubá
3935,A5106821649,Ícaro Viterbre Debique Sousa,Não identificado,"2023, 2024",", Doutorado em Engenharia Aeronáutica e Mecâni..."
3936,A5002602597,Ítalo de Abreu Gonçalves,Não identificado,"2020, 2021","Academic Unit of Engineering of Itabira, Feder..."


In [ ]:
unifei_df.to_csv(
    "/content/pesquisadores_unifei_todos_institutos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo: /content/pesquisadores_unifei_todos_institutos.csv")


Arquivo salvo: /content/pesquisadores_unifei_todos_institutos.csv


In [ ]:
from google.colab import files

files.download("/content/pesquisadores_unifei_todos_institutos.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Um CSV separado por instituto

Pesquisadores com mais de um instituto aparecem em mais
de um arquivo; os "Não identificado" ficam de fora (revise a coluna `afiliacoes` desses casos).

In [ ]:
import os

pasta_saida = "/content/pesquisadores_por_instituto"
os.makedirs(pasta_saida, exist_ok=True)

for sigla in INSTITUTOS_UNIFEI:

    df_instituto = unifei_df[
        unifei_df["instituto"].str.contains(sigla, na=False)
    ]

    if len(df_instituto) == 0:
        continue

    caminho = f"{pasta_saida}/pesquisadores_{sigla}.csv"

    df_instituto.to_csv(
        caminho,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"{sigla}: {len(df_instituto)} pesquisadores -> {caminho}")

print("\nNão identificado:", (unifei_df["instituto"] == "Não identificado").sum())


IEM: 199 pesquisadores -> /content/pesquisadores_por_instituto/pesquisadores_IEM.csv
IEPG: 135 pesquisadores -> /content/pesquisadores_por_instituto/pesquisadores_IEPG.csv
IESTI: 96 pesquisadores -> /content/pesquisadores_por_instituto/pesquisadores_IESTI.csv
IFQ: 233 pesquisadores -> /content/pesquisadores_por_instituto/pesquisadores_IFQ.csv
IMC: 87 pesquisadores -> /content/pesquisadores_por_instituto/pesquisadores_IMC.csv
IRN: 248 pesquisadores -> /content/pesquisadores_por_instituto/pesquisadores_IRN.csv
ISEE: 138 pesquisadores -> /content/pesquisadores_por_instituto/pesquisadores_ISEE.csv
IEI: 68 pesquisadores -> /content/pesquisadores_por_instituto/pesquisadores_IEI.csv

Não identificado: 2780


In [ ]:
import shutil

shutil.make_archive("/content/pesquisadores_por_instituto", "zip", pasta_saida)

files.download("/content/pesquisadores_por_instituto.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Rede de coautoria (todos os pesquisadores, todos os institutos juntos)

In [ ]:
import itertools
from collections import defaultdict

pesquisadores_validos = set(
    unifei_df["author_id"].astype(str)
)

colaboracoes = defaultdict(int)

for work in trabalhos:

    autores = []

    for authorship in work.get("authorships", []):

        author = authorship.get("author")

        if not author:
            continue

        author_id = author.get("id")

        if not author_id:
            continue

        author_id = author_id.split("/")[-1]

        if author_id in pesquisadores_validos:
            autores.append(author_id)

    autores = list(set(autores))

    for autor_a, autor_b in itertools.combinations(sorted(autores), 2):
        colaboracoes[(autor_a, autor_b)] += 1


edges = []

for (source, target), weight in colaboracoes.items():
    edges.append({
        "source": source,
        "target": target,
        "weight": weight
    })

edges_df = pd.DataFrame(edges)

edges_df = edges_df.sort_values(
    "weight", ascending=False
).reset_index(drop=True)

print("Número de arestas:", len(edges_df))
display(edges_df.head(20))

edges_df.to_csv(
    "/content/edges.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nArquivo salvo: /content/edges.csv")


Número de arestas: 11410


,source,target,weight
0,A5011550028,A5059943716,51
1,A5011550028,A5047492027,48
2,A5034546401,A5042124823,44
3,A5042124823,A5073459961,39
4,A5034546401,A5073459961,38
5,A5047492027,A5059943716,38
6,A5083299566,A5085295667,37
7,A5021084634,A5035016214,33
8,A5017902432,A5034546401,29
9,A5026069948,A5073340466,29



Arquivo salvo: /content/edges.csv


In [ ]:
# ============================================================
# CRIAR NODES (com o instituto de cada pesquisador)
# ============================================================

nodes_df = unifei_df[["author_id", "name", "instituto"]].copy()

nodes_df = nodes_df.rename(
    columns={"author_id": "id", "name": "label"}
)

nodes_df.to_csv(
    "/content/nodes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Número de nós:", len(nodes_df))
print("Arquivo salvo: /content/nodes.csv")

display(nodes_df.head())


Número de nós: 3938
Arquivo salvo: /content/nodes.csv


,id,label,instituto
0,A5000276221,ADRIELLE CRISTINA SASSA,IEI
1,A5070765202,Alisson Clementino Da Silva,IEI
2,A5045443140,Ana Carolina Moreira Silva,IEI
3,A5039088323,Ana Carolina Oliveira Santos,IEI
4,A5033167027,Ana Cristina Tolentino Cabral,IEI


In [ ]:
files.download("/content/nodes.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("/content/edges.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np

nodes = pd.read_csv("/content/nodes.csv")
edges = pd.read_csv("/content/edges.csv")

ids = nodes["id"].astype(str).tolist()

id_to_index = {
    author_id: i
    for i, author_id in enumerate(ids)
}

n = len(ids)

matriz = np.zeros((n, n), dtype=int)

for _, row in edges.iterrows():

    source = str(row["source"])
    target = str(row["target"])
    weight = int(row["weight"])

    i = id_to_index[source]
    j = id_to_index[target]

    matriz[i][j] = weight
    matriz[j][i] = weight

np.savetxt(
    "/content/unifei_colaboracao_2020_2026.txt",
    matriz,
    fmt="%d"
)

print("Dataset TXT criado!")
print("Dimensão:", matriz.shape)
print("Arquivo: /content/unifei_colaboracao_2020_2026.txt")


Dataset TXT criado!
Dimensão: (3938, 3938)
Arquivo: /content/unifei_colaboracao_2020_2026.txt


In [ ]:
files.download("/content/unifei_colaboracao_2020_2026.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>